In [2]:
# ==========================================
# LAB 11: GPT-2 Fine-Tuning (CORRECTED)
# ==========================================

!pip install -q transformers datasets accelerate

import torch
from transformers import (
    GPT2LMHeadModel,
    GPT2Tokenizer,
    Trainer,
    TrainingArguments,
    DataCollatorForLanguageModeling
)
from datasets import Dataset

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", device)

# ==================================================
# PRODUCT REVIEW DATASET
# ==================================================
review_corpus = [
    "this phone has an amazing battery life and the camera quality is outstanding for the price.",
    "i bought this laptop for college and it handles all my assignments and coding projects perfectly.",
    "the sound quality of these headphones is incredible with deep bass and clear vocals.",
    "this smartwatch tracks my steps accurately and the heart rate monitor is very reliable.",
    "great wireless earbuds with noise cancellation that blocks out all background sound.",
    "the keyboard feels very comfortable for long typing sessions and the backlight is a nice touch.",
    "this portable charger saved me during travel and it charges my phone three times on a single charge.",
    "the tablet screen is bright and colorful which makes watching movies a great experience.",
    "i love this fitness tracker because it motivates me to reach my daily exercise goals.",
    "this bluetooth speaker is compact but delivers surprisingly loud and clear audio."
]

# ==================================================
# LOAD GPT-2
# ==================================================
tokenizer = GPT2Tokenizer.from_pretrained("gpt2")
tokenizer.pad_token = tokenizer.eos_token

model = GPT2LMHeadModel.from_pretrained("gpt2").to(device)

# ==================================================
# BEFORE FINE-TUNING
# ==================================================
print("\n===== BEFORE FINE-TUNING =====")

prompt = "this phone"
inputs = tokenizer(prompt, return_tensors="pt").to(device)

output = model.generate(
    **inputs,
    max_length=50,
    do_sample=True,
    temperature=0.7
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

# ==================================================
# TOKENIZE DATASET
# ==================================================
dataset = Dataset.from_dict({"text": review_corpus})

def tokenize_function(example):
    tokens = tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=64
    )
    tokens["labels"] = tokens["input_ids"].copy()
    return tokens

tokenized_dataset = dataset.map(tokenize_function)

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

# ==================================================
# TRAINING ARGUMENTS (FIXED)
# ==================================================
training_args = TrainingArguments(
    output_dir="./product_review_model",
    num_train_epochs=3,
    per_device_train_batch_size=2,
    logging_steps=5,
    save_strategy="no",
    report_to="none"
)

# ==================================================
# TRAINER
# ==================================================
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset,
    data_collator=data_collator
)

# ==================================================
# FINE-TUNING
# ==================================================
trainer.train()

# ==================================================
# AFTER FINE-TUNING
# ==================================================
print("\n===== AFTER FINE-TUNING =====")

inputs = tokenizer(prompt, return_tensors="pt").to(device)

output = model.generate(
    **inputs,
    max_length=50,
    do_sample=True,
    temperature=0.7
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

Using device: cpu


Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



===== BEFORE FINE-TUNING =====
this phone) which was completely useless. I had no idea what I was going to do with this phone or what I was going to do with any other phone, I was like, "I don't know what's going on here."




Map:   0%|          | 0/10 [00:00<?, ? examples/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Step,Training Loss
5,3.878338
10,2.745455
15,2.412957


Setting `pad_token_id` to `eos_token_id`:50256 for open-end generation.



===== AFTER FINE-TUNING =====
this phone is great and very comfortable. Love the phone. Love my husband loves it so much. I had my phone for a week and it was my first experience with my new phone. I love my pocket phones so much and feel comfortable with it
